# Silver Layer — Patients

## Source
`cdr_mimiciv.bronze.patients`

## Target
`cdr_mimiciv.silver.patients`

The target table is published with CREATE OR REPLACE semantics; downstream Gold depends on it.

## Grain
One row represents one unique patient identified by `subject_id`.

## Source-file evidence (local MIMIC-IV CSV, 364,627 rows)
`gender` contains only `F` (191,984) and `M` (172,643), already upper-case and trimmed, so the upper/trim rule is a no-op on that
release (kept as harmless whitespace standardisation). `anchor_age` has no NULLs.

## Silver objective
Create a reusable and clinically interpretable patient entity while preserving
MIMIC-IV de-identification semantics.

## Engineering principles
- Preserve source clinical meaning.
- Do not impute missing clinical values.
- Do not remove records unless a validated rule requires it.
- Preserve MIMIC-IV age de-identification: per the official documentation, a patient whose
  age in `anchor_year` is over 89 has `anchor_age` set to 91, so `anchor_age = 91` means
  "over 89, true age not available".
- `dod` is NULL when no death is recorded, but out-of-hospital deaths are only captured up
  to one year after hospital discharge (later deaths are censored), so NULL `dod` does not
  establish survival.
- Apply only deterministic, reusable Silver transformations.
- Defer analytical groupings such as age bands to Gold.

In [0]:
from pyspark.sql import functions as F
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

TARGET = f"{SILVER}.patients"

In [0]:
patients_bronze = spark.table(f"{BRONZE}.patients")

display(patients_bronze.limit(20))

In [0]:
%sql
Select count(*) 
from cdr_mimiciv.silver.patients
where age_deidentified_90_plus=true;

In [0]:
patient_profile = patients_bronze.select(
    F.count("*").alias("total_rows"),
    F.countDistinct("subject_id").alias("unique_subject_ids"),
    F.sum(
        F.when(F.col("subject_id").isNull(), 1).otherwise(0)
    ).alias("null_subject_ids")
)

display(patient_profile)

In [0]:
duplicate_patients = (
    patients_bronze
    .groupBy("subject_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate subject_ids:", duplicate_patients.count())

In [0]:
# check missing values
display(
    patients_bronze.select(
        *[
            F.sum(
                F.when(F.col(c).isNull(), 1).otherwise(0)
            ).alias(c)
            for c in [
                "subject_id",
                "gender",
                "anchor_age",
                "anchor_year",
                "anchor_year_group",
                "dod"
            ]
        ]
    )
)

In [0]:
# inspect rescued data (only if the Bronze loader created the column)
if "_rescued_data" in patients_bronze.columns:
    rescued_count = (
        patients_bronze
        .filter(F.col("_rescued_data").isNotNull())
        .count()
    )
    print("Rows with rescued data:", rescued_count)
else:
    print("No _rescued_data column in Bronze table; skipped.")

In [0]:
display(
    patients_bronze
    .groupBy("gender")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    patients_bronze.select(
        F.min("anchor_age").alias("min_anchor_age"),
        F.max("anchor_age").alias("max_anchor_age"),
        F.sum(
            F.when(F.col("anchor_age") == 91, 1).otherwise(0)
        ).alias("age_91_count")
    )
)

In [0]:
patients_silver = (
    patients_bronze
    .select(
        F.col("subject_id")
            .cast("int")
            .alias("subject_id"),

        F.upper(
            F.trim(F.col("gender"))
        ).alias("gender"),

        F.col("anchor_age")
            .cast("int")
            .alias("anchor_age"),

        F.col("anchor_year")
            .cast("int")
            .alias("anchor_year"),

        F.trim(
            F.col("anchor_year_group")
        ).alias("anchor_year_group"),

        # NOTE: the documentation types dod as TIMESTAMP(0) (described as a date of death). In the local
        # source CSV all 38,301 non-empty dod values are plain YYYY-MM-DD dates, so the date cast loses
        # nothing for that release (see `docs/gold_governance_checks.md`, Source readiness); recheck on other releases.
        F.col("dod")
            .cast("date")
            .alias("dod")
    )

    # Preserve the meaning of MIMIC's age de-identification
    .withColumn(
        "age_deidentified_90_plus",
        F.col("anchor_age") == 91
    )

    # Real technical processing timestamp
    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

In [0]:
# pre-publish validation
bronze_rows = patients_bronze.count()
pre = patients_silver.select(
    F.count("*").alias("silver_rows"),
    F.countDistinct("subject_id").alias("unique_subject_ids"),
    F.sum(F.when(F.col("subject_id").isNull(), 1).otherwise(0)).alias("null_subject_ids"),
    F.sum(F.when(F.col("anchor_age").isNull(), 1).otherwise(0)).alias("null_anchor_age"),
    F.sum(F.when(F.col("age_deidentified_90_plus").isNull(), 1).otherwise(0)).alias("null_age_flag"),
).first()
display(patients_silver.limit(5))

assert pre["silver_rows"] == bronze_rows, "Row count differs from Bronze"
assert pre["null_subject_ids"] == 0, "NULL subject_id present"
assert pre["unique_subject_ids"] == pre["silver_rows"], "subject_id is not unique"
assert pre["null_age_flag"] == pre["null_anchor_age"], "Flag NULL only where anchor_age is NULL"

In [0]:
# publish (CREATE OR REPLACE; downstream Gold depends on this table)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")
patients_silver.writeTo(TARGET).using("delta").createOrReplace()

print(f"Published: {TARGET}")

In [0]:
# post-publish validation of the published table
EXPECTED_SCHEMA = {
    "subject_id": "int",
    "gender": "string",
    "anchor_age": "int",
    "anchor_year": "int",
    "anchor_year_group": "string",
    "dod": "date",
    "age_deidentified_90_plus": "boolean"
}

published = spark.table(TARGET)
actual_schema = {f.name: f.dataType.simpleString() for f in published.schema.fields}
assert list(actual_schema.items()) == list(EXPECTED_SCHEMA.items()), (
    f"Schema mismatch: {actual_schema}"
)

post = published.select(
    F.count("*").alias("rows"),
    F.countDistinct("subject_id").alias("unique_ids"),
    F.sum(F.when(F.col("subject_id").isNull(), 1).otherwise(0)).alias("null_ids"),
    F.sum(F.when(F.col("age_deidentified_90_plus").isNull(), 1).otherwise(0)).alias("null_age_flag"),
).first()

assert post["rows"] == bronze_rows, "Published row count differs from Bronze"
assert post["null_ids"] == 0
assert post["unique_ids"] == post["rows"]
assert post["null_age_flag"] == pre["null_anchor_age"]
print("Validated:", post.asDict())

In [0]:
dbutils.notebook.exit("patients:OK")